# Pertemuan 5 — Praktikum 3: DBSCAN pada Data Buatan

DBSCAN mengelompokkan titik berdasarkan kepadatan. Berbeda dari K-Means, kita tidak menentukan jumlah klaster di awal. Titik yang tidak masuk kelompok diberi label `-1` atau noise.

Sumber: [Praktikum 3](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-2025-2026/js04-klasterisasi/praktikum-3). Bagian kode yang kosong pada modul sudah dilengkapi.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn import metrics


## 1. Membuat dan menstandarkan data
Kita membuat 750 titik dengan tiga pusat kelompok. StandardScaler mengubah setiap fitur agar rata-ratanya mendekati 0 dan simpangan bakunya mendekati 1. Langkah ini membuat skala jarak lebih seimbang.


In [2]:
centers = [[1, 1], [-1, -1], [1, -1]]
X_raw, labels_true = make_blobs(n_samples=750, centers=centers,
                               cluster_std=0.4, random_state=0)
X = StandardScaler().fit_transform(X_raw)
plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], s=20)
plt.title('Data Buatan Sebelum DBSCAN')
plt.xlabel('Fitur 1 (standar)')
plt.ylabel('Fitur 2 (standar)')
plt.show()


## 2. Menjalankan DBSCAN
`eps=0.3` adalah radius untuk mencari tetangga. `min_samples=10` adalah jumlah minimum titik dalam radius tersebut, termasuk titik itu sendiri, agar menjadi core sample. Noise tidak dihitung sebagai klaster.


In [3]:
db = DBSCAN(eps=0.3, min_samples=10).fit(X)
labels = db.labels_
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int(np.sum(labels == -1))
print('Jumlah klaster:', n_clusters)
print('Jumlah titik noise:', n_noise)


Jumlah klaster: 3
Jumlah titik noise: 18


## 3. Mengevaluasi hasil
Label asli hanya digunakan untuk evaluasi, bukan untuk melatih DBSCAN.

- Homogeneity: apakah isi setiap klaster berasal dari satu label asli.
- Completeness: apakah data dengan label asli sama berada pada klaster yang sama.
- V-measure: gabungan homogeneity dan completeness.
- ARI dan AMI: kesesuaian pengelompokan dengan label asli setelah koreksi peluang.
- Silhouette: membandingkan jarak dalam kelompok dengan jarak ke kelompok lain.

Mengikuti contoh modul, Silhouette di sini menghitung semua titik, termasuk noise sebagai satu label. Karena noise bukan klaster biasa, kita juga menampilkan nilai tanpa noise sebagai pembanding.


In [4]:
hasil = {
    'Homogeneity': metrics.homogeneity_score(labels_true, labels),
    'Completeness': metrics.completeness_score(labels_true, labels),
    'V-measure': metrics.v_measure_score(labels_true, labels),
    'ARI': metrics.adjusted_rand_score(labels_true, labels),
    'AMI': metrics.adjusted_mutual_info_score(labels_true, labels),
    'Silhouette (semua titik)': metrics.silhouette_score(X, labels),
}
mask = labels != -1
hasil['Silhouette (tanpa noise)'] = metrics.silhouette_score(X[mask], labels[mask])
display(pd.DataFrame(hasil.items(), columns=['Metrik', 'Nilai']).round(3))


## 4. Melihat hasil klasterisasi
Titik besar berwarna adalah core sample. Titik kecil berwarna adalah titik tepi yang tetap termasuk klaster. Titik hitam adalah noise.


In [5]:
def plot_dbscan(X, db, ax, title):
    labels = db.labels_
    core_mask = np.zeros_like(labels, dtype=bool)
    core_mask[db.core_sample_indices_] = True
    unique_labels = sorted(set(labels))
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    for label, color in zip(unique_labels, colors):
        if label == -1:
            color = 'black'
        member = labels == label
        core = member & core_mask
        border = member & ~core_mask
        ax.scatter(X[core, 0], X[core, 1], c=[color], s=70, edgecolors='k', linewidths=0.3)
        ax.scatter(X[border, 0], X[border, 1], c=[color], s=15, edgecolors='k', linewidths=0.3)
    ax.set_title(title)
    ax.set_xlabel('Fitur 1 (standar)')
    ax.set_ylabel('Fitur 2 (standar)')


In [6]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_dbscan(X, db, ax, f'Hasil DBSCAN: {n_clusters} Klaster dan {n_noise} Noise')
plt.show()


## Kesimpulan
DBSCAN menemukan tiga klaster pada data buatan ini dan menandai titik yang berada di daerah kurang padat sebagai noise. Nilai evaluasi membantu melihat kesesuaian dengan kelompok asli. Hasil DBSCAN dipengaruhi radius `eps` dan jumlah minimum tetangga.
